In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

base_path = Path(r"C:\FeedbackControl\pyclm\data\mdck\rotations")

track_path = base_path / "tracks"
save_path = Path(r"C:\FeedbackControl\pyclm\figures\New Vortex Figure")

import matplotlib as mpl

mpl.rc("axes", facecolor="#ffffff00", grid=False, edgecolor="k", labelcolor="k")
mpl.rc("figure", facecolor="#00000000", dpi=100)
mpl.rc(
    "axes.spines",
    top=True,
    right=True,
)
mpl.rc("xtick", color="k", bottom=True)
mpl.rc("ytick", color="k", left=True)

In [ ]:
tracks = {}
stems = []
files = []

for f in track_path.glob("*.csv"):
    files.append(f)
    stems.append(f.stem)
    tracks[f.stem] = pd.read_csv(f)


px_um = 1.33
frame_min = 5
img_shape = (800, 800)

for k, df in enumerate(tracks.values()):
    invert_y_axis = False
    if stems[k].startswith("rotate"):
        invert_y_axis = True

    if invert_y_axis:
        df["px_y"] = img_shape[0] - df["px_y"]

    df["px_x_centered"] = df["px_x"] - img_shape[1] / 2
    df["px_y_centered"] = df["px_y"] - img_shape[0] / 2

    df["um_x"] = df["px_x"] * px_um
    df["um_y"] = df["px_y"] * px_um

    df["um_x_centered"] = df["px_x_centered"] * px_um
    df["um_y_centered"] = df["px_y_centered"] * px_um

    df["time"] = df["frame"] * frame_min

In [ ]:
import cv2
import tifffile
from scipy.spatial.distance import cdist

k = 3

n_samples = 30
time = 160 * 5
time_limit = 60
sigma = 25

invert_y_axis = False
if stems[k].startswith("rotate"):
    invert_y_axis = True

print(stems[k])
df = tracks[stems[k]]

df["theta"] = np.arctan2(df["um_x_centered"], df["um_y_centered"])
df["track_initial_theta"] = df["track_id"].map(df.groupby("track_id")["theta"].first())
df["track_initial_theta_vec_x"] = np.cos(df["track_initial_theta"])
df["track_initial_theta_vec_y"] = np.sin(df["track_initial_theta"])
df["track_start_time"] = df.groupby("track_id")["time"].transform("min")

df["dx"] = df.groupby("track_id")["um_x"].diff() / frame_min
df["dy"] = df.groupby("track_id")["um_y"].diff() / frame_min
df["dtot"] = np.sqrt(df["dx"] ** 2 + df["dy"] ** 2)


df_cropped = df[
    df["time"].between(180, 900)
    & df["px_x"].between(150, 650)
    & df["px_y"].between(150, 650)
    & (df["dtot"] < 2)
    & (~df["dx"].isna())
    & (~df["dy"].isna())
].copy()

# get data points near time, filter out bad tracks
df = df[(df["time"] - time).abs() < time_limit]
df = df[~df["dx"].isna()]
df = df[~df["dy"].isna()]
df = df[df["dtot"] < 2].copy()
df = df[df["track_start_time"] < time - 200].copy()

# get sampling positions in image coordinates and um coordinates
ax = np.linspace(-img_shape[1] / 2, img_shape[1] / 2, n_samples) * px_um
ay = np.linspace(-img_shape[0] / 2, img_shape[0] / 2, n_samples) * px_um

px_x = np.linspace(0, img_shape[1], n_samples)
px_y = np.linspace(0, img_shape[0], n_samples)

xx, yy = np.meshgrid(ax, ay)
X = np.array([xx.flatten(), yy.flatten()]).T

px_xx, px_yy = np.meshgrid(px_x, px_y)
X_px = np.array([px_xx.flatten(), px_yy.flatten()])

# get dxdy at sample positions using gaussian kernels
pos = df[["um_x_centered", "um_y_centered"]].values
distances = cdist(X, pos)
weights = np.exp(-0.5 * np.square(distances) / np.square(sigma))


dx_weighted = np.mean(weights * df["dx"].values[None, :], axis=1) / np.mean(
    weights, axis=1
)
dy_weighted = np.mean(weights * df["dy"].values[None, :], axis=1) / np.mean(
    weights, axis=1
)
theta = np.arctan2(dx_weighted, dy_weighted)
area_weighted = np.mean(weights * df["area"].values[None, :], axis=1) / np.mean(
    weights, axis=1
)
initial_theta_x_weighted = np.mean(
    weights * df["track_initial_theta_vec_x"].values[None, :], axis=1
) / np.mean(weights, axis=1)
initial_theta_y_weighted = np.mean(
    weights * df["track_initial_theta_vec_y"].values[None, :], axis=1
) / np.mean(weights, axis=1)
initial_theta_weighted = np.arctan2(initial_theta_y_weighted, initial_theta_x_weighted)

fig, ax = plt.subplots(figsize=(6, 6))

# show light bars
frame_data, frame_pattern = tifffile.imread(base_path / f"{stems[k][:-7]}.tif")[
    int(time // frame_min)
].astype(int)

if invert_y_axis:
    frame_data = np.flip(frame_data, axis=0)
    frame_pattern = np.flip(frame_pattern, axis=0)

vmin, vmax = 1000, 7500
frame_data = np.clip((frame_data - vmin) / (vmax - vmin), 0, 1)
# ax.imshow(np.stack([frame_data, frame_data, frame_data], axis=-1))
# ax.imshow(np.stack([np.zeros_like(frame_pattern), frame_pattern, frame_pattern, frame_pattern // 2], axis=-1))

# plot velocities
q = ax.quiver(
    X_px[0],
    X_px[1],
    dx_weighted * 50,
    dy_weighted * 50,
    color="k",
    scale_units="xy",
    clim=(-np.pi, np.pi),
    angles="xy",
    scale=1,
)
# cbar = fig.colorbar(q, ax=ax, orientation='vertical')
# cbar.set_label('Theta')


ax.axis("equal")
ax.set_xlim(150, 650)
ax.set_ylim(150, 650)
ax.set_xticks([])
ax.set_yticks([])
ax.invert_yaxis()


plt.savefig(save_path / f"local_movement_{stems[k]}.pdf", dpi=300)
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
initial_theta_reshaped_x = initial_theta_x_weighted.reshape(n_samples, n_samples)
initial_theta_reshaped_y = initial_theta_y_weighted.reshape(n_samples, n_samples)

initial_theta_resized_x = cv2.resize(
    initial_theta_reshaped_x, img_shape, interpolation=cv2.INTER_LINEAR
)
initial_theta_resized_y = cv2.resize(
    initial_theta_reshaped_y, img_shape, interpolation=cv2.INTER_LINEAR
)
initial_theta_resized = np.arctan2(initial_theta_resized_y, initial_theta_resized_x)
cmap = sns.color_palette("husl", as_cmap=True)
import colorcet as cc

cmap = cc.cm.CET_C10

initial_theta_colored = cmap((initial_theta_resized - (-np.pi)) / (2 * np.pi))
r = 1.0
initial_theta_colored = initial_theta_colored * r + (1 - r) * np.array([1, 1, 1, 1])
stacked_frame_data = np.stack([frame_data, frame_data, frame_data], axis=-1)
frame_colored = stacked_frame_data * initial_theta_colored[..., :3] + (
    1 - initial_theta_colored[..., :3]
) * np.array([1, 1, 1])
ax.imshow(frame_colored, extent=(0, 800, 0, 800), origin="lower", vmin=0, vmax=1)
# plt.imshow(initial_theta_weighted.reshape(n_samples, n_samples), extent=(0, 800, 0, 800), origin="lower", cmap="hsv", vmin=-np.pi, vmax=np.pi)
# plt.colorbar(label="Direction (radians)")
ax.set_xlim(150, 650)
ax.set_ylim(150, 650)
ax.set_xticks([])
ax.set_yticks([])
ax.invert_yaxis()
plt.savefig(save_path / f"cell_theta_source_{stems[k]}.png", dpi=300)

plt.show()

In [ ]:
k = 3

n_samples = 75
time = 160 * 5
time_limit = 60
sigma = 10

invert_y_axis = False
if stems[k].startswith("rotate"):
    invert_y_axis = True

print(stems[k])
df = tracks[stems[k]]

df["theta"] = np.arctan2(df["um_x_centered"], df["um_y_centered"])
df["track_initial_theta"] = df["track_id"].map(df.groupby("track_id")["theta"].first())
df["track_initial_theta_vec_x"] = np.cos(df["track_initial_theta"])
df["track_initial_theta_vec_y"] = np.sin(df["track_initial_theta"])
df["track_start_time"] = df.groupby("track_id")["time"].transform("min")

df["dx"] = df.groupby("track_id")["um_x"].diff() / frame_min
df["dy"] = df.groupby("track_id")["um_y"].diff() / frame_min
df["dtot"] = np.sqrt(df["dx"] ** 2 + df["dy"] ** 2)


df_cropped = df[
    df["time"].between(180, 900)
    & df["px_x"].between(150, 650)
    & df["px_y"].between(150, 650)
    & (df["dtot"] < 2)
    & (~df["dx"].isna())
    & (~df["dy"].isna())
].copy()

# get data points near time, filter out bad tracks
df = df[(df["time"] - time).abs() < time_limit]
df = df[~df["dx"].isna()]
df = df[~df["dy"].isna()]
df = df[df["dtot"] < 2].copy()
# df = df[df["track_start_time"] < time - 200].copy()

# get sampling positions in image coordinates and um coordinates
ax = np.linspace(-img_shape[1] / 2, img_shape[1] / 2, n_samples) * px_um
ay = np.linspace(-img_shape[0] / 2, img_shape[0] / 2, n_samples) * px_um

px_x = np.linspace(0, img_shape[1], n_samples)
px_y = np.linspace(0, img_shape[0], n_samples)

xx, yy = np.meshgrid(ax, ay)
X = np.array([xx.flatten(), yy.flatten()]).T

px_xx, px_yy = np.meshgrid(px_x, px_y)
X_px = np.array([px_xx.flatten(), px_yy.flatten()])

# get dxdy at sample positions using gaussian kernels
pos = df[["um_x_centered", "um_y_centered"]].values
distances = cdist(X, pos)
weights = np.exp(-0.5 * np.square(distances) / np.square(sigma))


dx_weighted = np.mean(weights * df["dx"].values[None, :], axis=1) / np.mean(
    weights, axis=1
)
dy_weighted = np.mean(weights * df["dy"].values[None, :], axis=1) / np.mean(
    weights, axis=1
)
theta = np.arctan2(dx_weighted, dy_weighted)
area_weighted = np.mean(weights * df["area"].values[None, :], axis=1) / np.mean(
    weights, axis=1
)
initial_theta_x_weighted = np.mean(
    weights * df["track_initial_theta_vec_x"].values[None, :], axis=1
) / np.mean(weights, axis=1)
initial_theta_y_weighted = np.mean(
    weights * df["track_initial_theta_vec_y"].values[None, :], axis=1
) / np.mean(weights, axis=1)
initial_theta_weighted = np.arctan2(initial_theta_y_weighted, initial_theta_x_weighted)

import colorstamps

rgb, stamp = colorstamps.apply_stamp(
    dx_weighted.reshape(n_samples, n_samples) * 60,
    dy_weighted.reshape(n_samples, n_samples) * 60,
    "peak",
    vmin_0=-40,
    vmax_0=40,
    vmin_1=-40,
    vmax_1=40,
)
fig, ax = plt.subplots(1, 1, figsize=(4, 4), dpi=300)
ax.imshow(rgb, extent=(0, 800, 0, 800))
# q = ax.quiver(X_px[0], X_px[1], dx_weighted*20, dy_weighted*20, color="k", scale_units="xy",angles="xy", scale=1)

ax.set_xlim(150, 650)
ax.set_ylim(150, 650)
ax.set_xticks([])
ax.set_yticks([])
ax.invert_yaxis()

plt.savefig(save_path / f"local_movement_colorquiver_{stems[k]}.pdf", dpi=300)
plt.show()

# also show colormap as in separate ax to illustrate functionality
fig, ax = plt.subplots(1, 1, figsize=(4, 4), dpi=300)

stamp.show_in_ax(ax)
ax.set_ylabel(r"$dy$")
ax.set_xlabel(r"$dx$")
ax.axis("off")
plt.savefig(save_path / f"local_movement_colorquiver_colormap_{stems[k]}.pdf", dpi=300)
plt.show()

In [ ]:
from collections import defaultdict


def plot_over_phase(ax, k_vals, metric="vertical speed mean", color="b"):
    avg_velocities = defaultdict(list)

    for k in k_vals:
        print(stems[k])
        df = tracks[stems[k]]

        df["radius"] = np.sqrt(df["um_x_centered"] ** 2 + df["um_y_centered"] ** 2)
        df["angle"] = np.arctan2(df["um_y_centered"], df["um_x_centered"])

        df["dx"] = df.groupby("track_id")["um_x"].diff() / frame_min
        df["dy"] = df.groupby("track_id")["um_y"].diff() / frame_min
        df["dtot"] = np.sqrt(df["dx"] ** 2 + df["dy"] ** 2)

        df["dradial"] = df["dx"] * np.cos(df["angle"]) + df["dy"] * np.sin(df["angle"])
        df["dradial_hr"] = df["dradial"] * 60
        df["dtangential"] = df["dx"] * np.sin(df["angle"]) - df["dy"] * np.cos(
            df["angle"]
        )
        df["dtangential_hr"] = df["dtangential"] * 60

        df_cropped = df[
            df["time"].between(180, 900)
            & df["px_x"].between(150, 650)
            & df["px_y"].between(150, 650)
            & (df["dtot"] < 2)
            & (~df["dx"].isna())
            & (~df["dy"].isna())
        ].copy()

        bw = 20
        df_cropped["radius_bin"] = ((df_cropped["radius"] - bw / 2) // bw) * bw + bw
        df_cropped = df_cropped[df_cropped["radius_bin"] <= 300]

        t = df_cropped.groupby("radius_bin")[metric].mean().reset_index()
        avg_velocities["radius_bin"].extend(t["radius_bin"])
        avg_velocities[metric].extend(t[metric])
        avg_velocities["replicate"].extend([stems[k].split(".")[2][:2]] * len(t))

    sns.lineplot(
        pd.DataFrame(avg_velocities),
        x="radius_bin",
        y=metric,
        errorbar=("sd", 1),
        color=color,
        lw=3,
        ax=ax,
        legend=False,
    )

In [ ]:
colors = {
    "FBC": "sandybrown",
    "open-loop": "lightcoral",
}

fig, ax = plt.subplots(figsize=(3, 3))

plot_over_phase(ax, [0, 1, 2], metric="dtangential_hr", color=colors["open-loop"])
plot_over_phase(ax, [3, 4, 5, 6], metric="dtangential_hr", color=colors["FBC"])

plt.axhline(0, color="k", linestyle="--")
plt.ylabel("Tissue Velocity (um/h)")
plt.xlabel("Radius")
# plt.xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi], ["-π", "-π/2", "0", "π/2", "π"])
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.xlim(0, 300)
plt.xlabel("Radius (um)")
plt.ylabel("Tangential Velocity (um / hr)")
plt.savefig(save_path / f"tangential_velocity_over_radius.pdf", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(3, 3))

plot_over_phase(ax, [0, 1, 2], metric="dradial_hr", color=colors["open-loop"])
plot_over_phase(ax, [3, 4, 5, 6], metric="dradial_hr", color=colors["FBC"])

plt.axhline(0, color="k", linestyle="--")
plt.ylabel("Tissue Velocity (um/h)")
plt.xlabel("Radius")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
# plt.xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi], ["-π", "-π/2", "0", "π/2", "π"])

plt.xlim(0, 300)
plt.xlabel("Radius (um)")
plt.ylabel("Radial Velocity (um / hr)")
plt.savefig(save_path / f"radial_velocity_over_radius.pdf", bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))
plt.ylim(-10, 10)
plt.xlim(0, 300)
# add second x axis for bar velocity, with ticks/labels on the inside
ax.spines["bottom"].set_position("zero")
pos = ax.spines["bottom"].get_position()
print(pos)
ax.secondary_xaxis(0.495, functions=(lambda x: 0.3 * x, lambda x: x / 0.3)).set_xlabel(
    ""
)

plot_over_phase(ax, [0, 1, 2], metric="dtangential_hr", color=colors["open-loop"])
# plot_over_phase(ax, [3, 4, 5, 6], metric="dtangential_hr", color=colors["FBC"])

# plt.axhline(0, color="k", linestyle="--")
plt.ylabel("Tissue Velocity (um/h)")
plt.xlabel("Radius")
# plt.xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi], ["-π", "-π/2", "0", "π/2", "π"])
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tick_params(axis="x", direction="in", pad=-15)


plt.xlabel("")
plt.ylabel("Tangential Velocity (um / hr)")
plt.savefig(
    save_path / f"bars_tangential_velocity_double_x_axis.pdf", bbox_inches="tight"
)
plt.show()

In [ ]:
for k in [3, 4, 5, 6]:
    time = 900
    time_limit = 7.5

    print(stems[k])
    df = tracks[stems[k]]

    df["radius"] = np.sqrt(df["um_x_centered"] ** 2 + df["um_y_centered"] ** 2)
    df["angle"] = np.arctan2(df["um_y_centered"], df["um_x_centered"])

    df["dx"] = df.groupby("track_id")["um_x"].diff() / frame_min
    df["dy"] = df.groupby("track_id")["um_y"].diff() / frame_min
    df["dtot"] = np.sqrt(df["dx"] ** 2 + df["dy"] ** 2)

    df["dradial"] = df["dx"] * np.cos(df["angle"]) + df["dy"] * np.sin(df["angle"])
    df["dtangential"] = -df["dx"] * np.sin(df["angle"]) + df["dy"] * np.cos(df["angle"])
    df["dtangential_hr"] = df["dtangential"] * 60

    df_cropped = df[
        df["time"].between(180, 900)
        & df["px_x"].between(150, 650)
        & df["px_y"].between(150, 650)
        & (df["dtot"] < 2)
        & (~df["dx"].isna())
        & (~df["dy"].isna())
    ].copy()

    df_cropped["radius_bin"] = (df_cropped["radius"] // 20) * 20

    sns.lineplot(
        df_cropped,
        x="radius_bin",
        y="dtangential_hr",
        errorbar=None,
        label=stems[k].split(".")[2][:2],
    )
plt.legend(title="replicate")
plt.xlim(0, 250)
plt.xlabel("Radius (um)")
plt.ylabel("Tangential Velocity (um / hr)")
plt.show()